# Notebook 03: Dialog Act Classifier

Fine-tunes RoBERTa-base on the Switchboard Dialog Act Corpus for dialog-act
classification. The trained model is used to compute the DA-based repair proxy
in the CGA analysis pipeline.

Outputs:
- Trained model saved for downstream use.
- Test accuracy (sw_acc) for the paper.

In [ ]:
!pip install -q transformers torch scikit-learn pandas tqdm accelerate

In [ ]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load Switchboard Dialog Act Data

In [ ]:
import pandas as pd

# Load SWDA directly from parquet files on HuggingFace (bypasses broken loading script)
BASE_URL = "https://huggingface.co/datasets/silicone/resolve/refs%2Fconvert%2Fparquet/swda"
train_df = pd.read_parquet(f"{BASE_URL}/train/0000.parquet")
val_df = pd.read_parquet(f"{BASE_URL}/validation/0000.parquet")
test_df = pd.read_parquet(f"{BASE_URL}/test/0000.parquet")

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")
print(f"Columns: {list(train_df.columns)}")
print(f"\nSample:\n{train_df.head(2)}")

# Build label mapping from the integer Label column
label_names = sorted(train_df["Label"].unique().tolist())
num_labels = len(label_names)
print(f"\nNumber of DA labels: {num_labels}")
print(f"Labels: {label_names[:10]}...")

In [ ]:
CONCILIATORY_TAGS = {"aa", "bk", "br", "ba"}

# Label column may be strings (tag names) or integers depending on parquet export
# Handle both cases
if train_df["Label"].dtype == object:
    # Labels are already strings
    label_to_idx = {name: idx for idx, name in enumerate(label_names)}
    idx_to_label = {idx: name for idx, name in enumerate(label_names)}
    conciliatory_indices = [label_to_idx[t] for t in CONCILIATORY_TAGS if t in label_to_idx]
else:
    # Labels are integers — we need the actual tag name mapping
    # The silicone/swda parquet uses integer labels; map them
    label_to_idx = {name: name for name in label_names}  # identity for int labels
    idx_to_label = {name: name for name in label_names}
    conciliatory_indices = [t for t in label_names if t in CONCILIATORY_TAGS]

for tag in CONCILIATORY_TAGS:
    if tag in label_to_idx:
        print(f"  Conciliatory tag '{tag}' -> index {label_to_idx[tag]}")
    else:
        print(f"  Conciliatory tag '{tag}' -> NOT FOUND in labels")

print(f"\nConciliatory indices: {conciliatory_indices}")
print(f"Label dtype: {train_df['Label'].dtype}")
print(f"First 5 labels: {train_df['Label'].head().tolist()}")

## 2. Prepare Data for Fine-tuning

In [ ]:
MODEL_NAME = "roberta-base"
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Detect column names (parquet may use "Utterance" or "utterance")
utt_col = "Utterance" if "Utterance" in train_df.columns else "utterance"
lab_col = "Label" if "Label" in train_df.columns else "label"
print(f"Using columns: utterance='{utt_col}', label='{lab_col}'")

# If labels are strings, encode them to integers
if train_df[lab_col].dtype == object:
    _label_to_int = {name: idx for idx, name in enumerate(label_names)}
    train_df["_label_int"] = train_df[lab_col].map(_label_to_int)
    val_df["_label_int"] = val_df[lab_col].map(_label_to_int)
    test_df["_label_int"] = test_df[lab_col].map(_label_to_int)
    int_lab_col = "_label_int"
else:
    int_lab_col = lab_col
    # Remap integer labels to contiguous 0..N-1
    _unique_labels = sorted(train_df[lab_col].unique().tolist())
    _label_remap = {old: new for new, old in enumerate(_unique_labels)}
    train_df["_label_int"] = train_df[lab_col].map(_label_remap)
    val_df["_label_int"] = val_df[lab_col].map(_label_remap)
    test_df["_label_int"] = test_df[lab_col].map(_label_remap)
    int_lab_col = "_label_int"
    # Update num_labels and label_names accordingly
    num_labels = len(_unique_labels)
    label_names = _unique_labels


class SWDADataset(Dataset):
    """PyTorch dataset for Switchboard Dialog Act utterances."""

    def __init__(self, df, tokenizer, max_length, utt_col, lab_col):
        self.texts = df[utt_col].tolist()
        self.labels = df[lab_col].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = SWDADataset(train_df, tokenizer, MAX_LENGTH, utt_col, int_lab_col)
val_dataset = SWDADataset(val_df, tokenizer, MAX_LENGTH, utt_col, int_lab_col)
test_dataset = SWDADataset(test_df, tokenizer, MAX_LENGTH, utt_col, int_lab_col)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"num_labels: {num_labels}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 3. Fine-tune RoBERTa

In [ ]:
# === [Colab only] Mount Google Drive for checkpoints — remove before submitting ===
from google.colab import drive
drive.mount("/content/drive")

CKPT_DIR = Path("/content/drive/MyDrive/phase-transition-amd/da_checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint directory: {CKPT_DIR}")

# Check for existing checkpoints
existing_ckpts = sorted(CKPT_DIR.glob("epoch_*.pt"))
if existing_ckpts:
    print(f"Found {len(existing_ckpts)} existing checkpoint(s): {[c.name for c in existing_ckpts]}")
    print("Training will resume from the latest one.")
else:
    print("No existing checkpoints. Training will start from scratch.")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

# Resume from checkpoint if available
start_epoch = 0
best_val_accuracy = 0.0
best_model_state = None

existing_ckpts = sorted(CKPT_DIR.glob("epoch_*.pt"))
if existing_ckpts:
    latest_ckpt = existing_ckpts[-1]
    print(f"Resuming from {latest_ckpt.name}...")
    checkpoint = torch.load(latest_ckpt, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch = checkpoint["epoch"]
    best_val_accuracy = checkpoint.get("best_val_accuracy", 0.0)
    if "best_model_state" in checkpoint:
        best_model_state = checkpoint["best_model_state"]
    print(f"Resumed after epoch {start_epoch}, best_val_acc={best_val_accuracy:.4f}")
else:
    print("Starting training from scratch.")

print(f"Total training steps: {total_steps}")
print(f"Epochs to run: {start_epoch + 1} -> {NUM_EPOCHS}")

In [ ]:
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Train]"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    avg_loss = total_loss / len(train_loader)

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Val]"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch + 1}: loss={avg_loss:.4f}, train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Save checkpoint to Google Drive after each epoch
    ckpt_path = CKPT_DIR / f"epoch_{epoch + 1}.pt"
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_accuracy": best_val_accuracy,
        "best_model_state": best_model_state,
        "train_acc": train_acc,
        "val_acc": val_acc,
    }, ckpt_path)
    print(f"  Checkpoint saved to {ckpt_path}")

model.load_state_dict(best_model_state)
print(f"\nBest validation accuracy: {best_val_accuracy:.4f}")

## 4. Evaluate on Test Set

In [ ]:
model.eval()
test_correct = 0
test_total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test evaluation"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=-1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

sw_acc = test_correct / test_total
print(f"sw_acc: {sw_acc:.4f}")

## 5. Save Model and Results

In [ ]:
SAVE_DIR = Path("../models/da_classifier")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

metadata = {
    "sw_acc": round(sw_acc, 4),
    "num_labels": num_labels,
    "label_names": label_names,
    "conciliatory_indices": conciliatory_indices,
    "model_name": MODEL_NAME,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "best_val_accuracy": round(best_val_accuracy, 4),
}

with open(SAVE_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)
with open(RESULTS_DIR / "da_classifier_results.json", "w") as f:
    json.dump({"sw_acc": round(sw_acc, 4)}, f, indent=2)

print(f"Model saved to {SAVE_DIR}")
print(f"sw_acc: {sw_acc:.4f}")

In [ ]:
# === [Colab only] Copy model and results to Google Drive — remove before submitting ===
import shutil

DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/phase-transition-amd/models/da_classifier")
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/phase-transition-amd/results")
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Copy saved model files
for f in SAVE_DIR.iterdir():
    shutil.copy2(f, DRIVE_MODEL_DIR / f.name)
    print(f"  Copied {f.name} -> {DRIVE_MODEL_DIR}")

# Copy results JSON
src_results = Path("../results/da_classifier_results.json")
if src_results.exists():
    shutil.copy2(src_results, DRIVE_RESULTS_DIR / src_results.name)
    print(f"  Copied {src_results.name} -> {DRIVE_RESULTS_DIR}")

print(f"\nModel saved to Google Drive: {DRIVE_MODEL_DIR}")
print(f"Results saved to Google Drive: {DRIVE_RESULTS_DIR}")